# Data Science Project: Draft Portfolio

## Business & Data Understanding
The primary business objective is to predict short-term price direction for Bitcoin (BTC/USD). This draft portfolio demonstrates the initial data pipeline, focusing on historical market data ingestion and exploratory technical features.

In [ ]:
import sys
import os
import warnings

# Append the project root to the system path to allow importing from the src/ directory
sys.path.append(os.path.abspath('..'))

from src.data_loader import fetch_btc_data
from src.features import engineer_features, transform_data
from src.models import train_logistic_baseline, train_xgboost
from src.evaluate import evaluate_model

warnings.filterwarnings('ignore')

## Data Preparation
We collect historical OHLCV data using the `yfinance` API. After fetching the data, we engineer several technical indicators: Returns, Simple Moving Averages (SMA), and Rolling Volatility. 

To handle potential dimensionality issues, we apply Z-score Normalization and Principal Component Analysis (PCA) to transform our feature space before modeling.

In [ ]:
# 1. Fetch raw market data
df_raw = fetch_btc_data(start_date='2022-01-01', end_date='2026-11-01')

# 2. Engineer features and define target variable
X, y = engineer_features(df_raw)

# 3. Sequential Train-Test Split (80/20) to prevent time-series data leakage
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 4. Data Transformation (Z-score Scaling & PCA)
X_train_pca, X_test_pca, pca_model = transform_data(X_train, X_test)

print(f"Data shape after PCA reduction: {X_train_pca.shape}")

## Modeling
We will evaluate two models:
1. **Logistic Regression:** A linear baseline model related to the regression concepts covered in lectures.
2. **XGBoost:** The advanced gradient boosting classifier specified in my project proposal.

In [ ]:
# Train baseline model
lr_model = train_logistic_baseline(X_train_pca, y_train)

# Train primary XGBoost classifier
xgb_model = train_xgboost(X_train_pca, y_train)

## Evaluation
We assess the performance of both models on the unseen test data using accuracy and classification reports.

In [ ]:
# Output metrics for both trained models
evaluate_model(lr_model, X_test_pca, y_test, model_name="Logistic Regression Baseline")
evaluate_model(xgb_model, X_test_pca, y_test, model_name="XGBoost Classifier")